<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/03-funciona-con-datos-nuevos/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 1.3 · ¿Funciona con datos nuevos?

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas tres celdas. `mensual` tiene una fila por cliente y mes, con su tipo. Usa siempre `random_state=42`, para poder comparar con las cifras de control.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def por_cliente_y_mes(ventas):
    """Una fila por cliente y mes en que compró, con sus líneas de pedido y su monto."""
    meses = ventas["fecha"].str[:7]
    return (
        ventas.groupby(["id_cliente", meses])
        .agg(lineas=("total", "size"), total=("total", "sum"))
        .reset_index()
    )

In [3]:
ventas = pd.read_csv(DATOS + "ventas.csv")
clientes = pd.read_csv(DATOS + "clientes.csv")

mensual = por_cliente_y_mes(ventas)
mensual = pd.merge(mensual, clientes[["id_cliente", "tipo"]], on="id_cliente", how="left")

mensual.shape

(6459, 5)

### 1. Separa y entrena

Filtra las filas de `Bodega` en `bodegas`. Separa `x` (`lineas`) e `y` (`total`) en entrenamiento y prueba, con 20 % para la prueba. Muestra cuántas filas quedan en cada parte: en prueba deben ser `772`.

Entrena una regresión **solo con entrenamiento**.

In [4]:
bodegas = mensual[mensual["tipo"] == "Bodega"]

x_entreno, x_prueba, y_entreno, y_prueba = train_test_split(
    bodegas[["lineas"]], bodegas["total"], test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(x_entreno, y_entreno)

print(len(x_entreno), len(x_prueba))

3086 772


### 2. El error en soles

Calcula el MAE en prueba, guárdalo en `error_recta`, y calcula también el de entrenamiento. El de prueba debe ser `358.95`. ¿Se parecen? ¿Qué significa? Responde en un comentario.

In [5]:
error_recta = mean_absolute_error(y_prueba, modelo.predict(x_prueba))

print(round(error_recta, 2))
print(round(mean_absolute_error(y_entreno, modelo.predict(x_entreno)), 2))

# Se parecen: la recta rinde casi igual con casos que no vio. No se aprendió las respuestas.

358.95
359.1


### 3. La línea base

Calcula el MAE que tendrías si predijeras para todos el promedio de `y_entreno`. Debe ser `786.05`. Guárdalo en `error_base`. ¿Cuánto mejora la recta, en porcentaje? Usa `(1 - error_recta / error_base) * 100`. Debe dar `54.3`.

In [6]:
promedio = y_entreno.mean()
error_base = (y_prueba - promedio).abs().mean()

print(round(error_base, 2))
print(round((1 - error_recta / error_base) * 100, 1))

786.05
54.3


### 4. El error de hacer trampa

Calcula el promedio con **toda** la tabla de bodegas, en lugar de solo con entrenamiento, y vuelve a medir el error de la línea base en prueba. Debe dar `785.56`. ¿Cambia mucho? Aunque aquí cambie poco, ¿por qué no se debe hacer? Responde en un comentario.

In [7]:
promedio_con_trampa = bodegas["total"].mean()

print(round((y_prueba - promedio_con_trampa).abs().mean(), 2))

# Cambia poco, pero el promedio ya miró los datos de prueba: dejan de ser casos nuevos.
# Con modelos más complejos, ese error puede hacer que un mal modelo parezca bueno.

785.56


### 5. Otra separación

En una celda, separa de nuevo con `random_state=7`, entrena otro modelo y calcula su MAE de prueba. Usa nombres nuevos, como `x_entreno_7`, para no pisar los anteriores. Debe dar `351.76`. ¿Es el mismo? ¿Qué te dice de confiar en una sola cifra? Responde en un comentario.

In [8]:
x_entreno_7, x_prueba_7, y_entreno_7, y_prueba_7 = train_test_split(
    bodegas[["lineas"]], bodegas["total"], test_size=0.2, random_state=7
)

modelo_7 = LinearRegression()
modelo_7.fit(x_entreno_7, y_entreno_7)

print(round(mean_absolute_error(y_prueba_7, modelo_7.predict(x_prueba_7)), 2))

# Cambia un poco: depende de qué filas cayeron en prueba. Una sola cifra es una estimación.

351.76


## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 1.4: Pronostica sin hacer trampa**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0